# 03 â€” Vector Search
## Databricks Expert Agent Project

**What this notebook does:**
1. Creates a Mosaic AI Vector Search endpoint (compute for similarity search)
2. Creates a Delta Sync index over `chatbot.rag_chatbot.doc_chunks`

**Output:** A live Vector Search index that automatically stays in sync with the doc_chunks Delta table. The agent uses this to retrieve the most relevant chunks for any question.

In [0]:
CATALOG       = "chatbot"
SCHEMA        = "rag_chatbot"
CHUNKS_TABLE  = f"{CATALOG}.{SCHEMA}.doc_chunks"

VS_ENDPOINT   = "databricks-expert-endpoint"   
VS_INDEX      = f"{CATALOG}.{SCHEMA}.doc_chunks_index"
EMBED_MODEL   = "databricks-gte-large-en"
EMBED_COLUMN  = "embedding"
ID_COLUMN     = "chunk_id"

print(f"Endpoint : {VS_ENDPOINT}")
print(f"Index    : {VS_INDEX}")
print(f"Source   : {CHUNKS_TABLE}")

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

try:
    vsc.create_endpoint(
        name=VS_ENDPOINT,
        endpoint_type="STANDARD"
    )
    print(f"âœ“ Endpoint '{VS_ENDPOINT}' created â€” waiting for it to be ready...")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"âœ“ Endpoint '{VS_ENDPOINT}' already exists â€” skipping creation")
    else:
        raise e

# Wait until endpoint is ONLINE before proceeding
import time
while True:
    status = vsc.get_endpoint(VS_ENDPOINT)["endpoint_status"]["state"]
    print(f"  Endpoint state: {status}")
    if status == "ONLINE":
        print("âœ“ Endpoint is ONLINE")
        break
    time.sleep(20)

In [0]:
spark.sql(f"ALTER TABLE {CHUNKS_TABLE} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
print(f"âœ“ Change Data Feed enabled on {CHUNKS_TABLE}")

# Verify it's on
props = spark.sql(f"SHOW TBLPROPERTIES {CHUNKS_TABLE}").filter("key = 'delta.enableChangeDataFeed'")
display(props)

In [0]:
# Run this to see your metastore details
display(spark.sql("SELECT * FROM system.information_schema.catalogs WHERE catalog_name = 'chatbot'"))

In [0]:
try:
    idx = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=VS_INDEX,
        source_table_name=CHUNKS_TABLE,
        pipeline_type="TRIGGERED",
        primary_key=ID_COLUMN,
        embedding_dimension=1024,
        embedding_vector_column=EMBED_COLUMN
    )
    print(f"âœ“ Index '{VS_INDEX}' created â€” waiting for initial sync...")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"âœ“ Index '{VS_INDEX}' already exists â€” skipping creation")
    else:
        raise e

# Wait for the index to finish its first sync
while True:
    status = vsc.get_index(VS_ENDPOINT, VS_INDEX).describe()
    state = status.get("status", {}).get("ready", False)
    detail = status.get("status", {}).get("message", "syncing...")
    print(f"  Index state: {detail}")
    if state:
        print("âœ“ Index is READY")
        break
    time.sleep(30)

In [0]:
import json

desc = vsc.get_index(VS_ENDPOINT, VS_INDEX).describe()
print(json.dumps(desc, indent=2))

In [0]:
import time, json

for _ in range(20):
    desc = vsc.get_index(VS_ENDPOINT, VS_INDEX).describe()
    status = desc["status"]

    progress = status.get("triggered_update_status", {}).get("triggered_update_progress", {})

    print("State:", status.get("detailed_state"))
    print("Indexed rows:", status.get("indexed_row_count"))
    print("Progress:", progress.get("sync_progress_completion"))
    print("Synced:", progress.get("num_synced_rows"), "/", progress.get("total_rows_to_sync"))
    print("-" * 80)

    if not progress:
        print("No active sync progress found.")
        break

    if progress.get("sync_progress_completion", 0) >= 0.999:
        print("Sync looks complete.")
        break

    time.sleep(30)

In [0]:
display(spark.sql("""
SELECT source, source_type, COUNT(*) AS chunks
FROM chatbot.rag_chatbot.doc_chunks
GROUP BY source, source_type
ORDER BY chunks DESC
"""))

In [0]:
%skip
idx = vsc.get_index(VS_ENDPOINT, VS_INDEX)

print("Triggering Vector Search index sync...")
idx.sync()

print("âœ“ Sync triggered")